# Guardrailer Embedding Model Evaluation
## Linear Probes on 4 Embedding Models (10,000 Samples)

**Protocol:** 10,000 stratified samples → 80/20 split (8,000 train / 2,000 test). All 4 models evaluated on the same data.
**Linear Probe:** LogisticRegression(max_iter=1000, C=1.0, random_state=42)
**Runtime:** Enable GPU (T4) in Kaggle Settings → Accelerator → GPU T4
**Checkpointing:** All progress auto-saves to /kaggle/working/checkpoints/

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas numpy matplotlib seaborn

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory: {props.total_memory / 1e9:.1f} GB")

CUDA_WORKS = False
if torch.cuda.is_available():
    try:
        _t = torch.randn(10).cuda()
        _t = _t * 2
        del _t
        torch.cuda.empty_cache()
        CUDA_WORKS = True
        print("CUDA kernel test: PASSED")
    except Exception as e:
        print(f"CUDA kernel test: FAILED ({e})")
else:
    print("No GPU detected.")

DEVICE = "cuda" if CUDA_WORKS else "cpu"
print(f"Using device: {DEVICE}")

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix, roc_curve
from sentence_transformers import SentenceTransformer

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 150

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
DATA_PATH = "/kaggle/input/datasets/prashannadeveloper/guardrailer-dataset-v1/guardrailer_dataset_v1.parquet"

df = pd.read_parquet(DATA_PATH)
df["text"] = df["prompt_text"]
df["label"] = df["is_malicious"].astype(int)

print(f"Total samples: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")

# Stratified 10,000 sample subset
subset, _ = train_test_split(df, train_size=10000, random_state=42, stratify=df["label"])

# Single 80/20 train/test split — shared across all 4 models
sub_train, sub_test = train_test_split(subset, test_size=0.20, random_state=42, stratify=subset["label"])

X_train = sub_train["text"].astype(str).tolist()
y_train = sub_train["label"].values
X_test = sub_test["text"].astype(str).tolist()
y_test = sub_test["label"].values

print(f"\nSubset: {len(subset)} samples")
print(f"Train: {len(X_train)}  Test: {len(X_test)}")
print(f"Train dist: safe={sum(y_train==0)}, malicious={sum(y_train==1)}")
print(f"Test dist:  safe={sum(y_test==0)}, malicious={sum(y_test==1)}")

In [ ]:
MODELS = [
    "BAAI/bge-large-en-v1.5",
    "sentence-transformers/all-MiniLM-L6-v2",
    "intfloat/e5-small-v2",
    "BAAI/bge-small-en-v1.5",
]

all_results = []

for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*60}")

    model = SentenceTransformer(model_name)
    print(f"  Model loaded. Device: {DEVICE}")

    start = time.time()
    print(f"  Encoding train set ({len(X_train)} samples)...")
    train_emb = model.encode(X_train, batch_size=64, show_progress_bar=True,
                             normalize_embeddings=True, device=DEVICE)
    train_emb = np.array(train_emb)

    print(f"  Encoding test set ({len(X_test)} samples)...")
    test_emb = model.encode(X_test, batch_size=64, show_progress_bar=True,
                            normalize_embeddings=True, device=DEVICE)
    test_emb = np.array(test_emb)
    encode_time = time.time() - start
    print(f"  Encoding time: {encode_time:.1f}s")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Centroid similarity
    centroid_safe = train_emb[y_train == 0].mean(axis=0)
    centroid_malicious = train_emb[y_train == 1].mean(axis=0)
    centroid_similarity = float(np.dot(centroid_safe, centroid_malicious))

    test_safe_emb = test_emb[y_test == 0]
    test_mal_emb = test_emb[y_test == 1]
    test_centroid_sim = float(np.dot(test_safe_emb.mean(axis=0), test_mal_emb.mean(axis=0)))

    print(f"  Centroid Similarity (train): {centroid_similarity:.6f}")
    print(f"  Centroid Similarity (test):  {test_centroid_sim:.6f}")

    # Linear probe
    print("  Training linear probe...")
    probe = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    probe.fit(train_emb, y_train)
    preds = probe.predict(test_emb)
    proba = probe.predict_proba(test_emb)[:, 1]

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    auc = roc_auc_score(y_test, proba)
    cm = confusion_matrix(y_test, preds)

    print(f"  Linear Probe Results:")
    print(f"    Accuracy:  {acc:.4f}")
    print(f"    F1-Score:  {f1:.4f}")
    print(f"    Precision: {prec:.4f}")
    print(f"    Recall:    {rec:.4f}")
    print(f"    AUC-ROC:   {auc:.4f}")

    # ROC data for plotting
    fpr, tpr, _ = roc_curve(y_test, proba)

    result = {
        "model_name": model_name,
        "short_name": model_name.split("/")[-1],
        "centroid_similarity_train": centroid_similarity,
        "centroid_similarity_test": test_centroid_sim,
        "accuracy": float(acc),
        "f1": float(f1),
        "precision": float(prec),
        "recall": float(rec),
        "auc_roc": float(auc),
        "confusion_matrix": cm.tolist(),
        "encode_time_sec": float(encode_time),
        "fpr": fpr,
        "tpr": tpr,
    }
    all_results.append(result)

print(f"\nAll {len(all_results)} models evaluated.")

In [ ]:
# Figure 1: Centroid Similarity
fig, ax = plt.subplots(figsize=(9, 5))
names = [r["short_name"] for r in all_results]
x = np.arange(len(names))
width = 0.35
ax.bar(x - width/2, [r["centroid_similarity_train"] for r in all_results], width,
       label="Train", color="#2196F3", alpha=0.8)
ax.bar(x + width/2, [r["centroid_similarity_test"] for r in all_results], width,
       label="Test", color="#FF5722", alpha=0.8)
ax.axhline(y=0.85, color="green", linestyle="--", alpha=0.5, label="Threshold (0.85)")
ax.set_ylabel("Cosine Similarity")
ax.set_title("Inter-Class Centroid Similarity by Embedding Model")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15)
ax.legend()
ax.set_ylim(0.5, 1.05)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + "fig_centroid_similarity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 2: Linear Probe Performance (grouped bar chart)
fig, ax = plt.subplots(figsize=(10, 5))
metrics = ["accuracy", "f1", "precision", "recall", "auc_roc"]
metric_labels = ["Accuracy", "F1-Score", "Precision", "Recall", "AUC-ROC"]
colors = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0", "#F44336"]
x = np.arange(len(names))
width = 0.15
for i, (metric, label, color) in enumerate(zip(metrics, metric_labels, colors)):
    values = [r[metric] for r in all_results]
    ax.bar(x + i * width - 2*width, values, width, label=label, color=color, alpha=0.8)
ax.set_ylabel("Score")
ax.set_title("Linear Probe Performance by Embedding Model")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15)
ax.legend(loc="lower right")
ax.set_ylim(0.4, 1.05)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + "fig_linear_probe_performance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 3: ROC Curves (all 4 models)
fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ["#F44336", "#2196F3", "#4CAF50", "#FF9800"]
for r, color in zip(all_results, colors_roc):
    ax.plot(r["fpr"], r["tpr"], color=color, linewidth=2,
            label=f'{r["short_name"]} (AUC={r["auc_roc"]:.4f})')
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves: Linear Probe on Embedding Space")
ax.legend(loc="lower right")
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
plt.savefig(OUTPUT_DIR + "fig_roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Summary table
summary_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ["fpr", "tpr", "confusion_matrix"]}
                           for r in all_results])
print(summary_df[["short_name", "centroid_similarity_test", "accuracy", "f1", "auc_roc"]].to_string(index=False))

# Save
summary_df.to_csv(OUTPUT_DIR + "embedding_results_summary.csv", index=False)
with open(OUTPUT_DIR + "embedding_results_summary.json", "w") as f:
    json.dump([{k: v for k, v in r.items() if k not in ["fpr", "tpr"]} for r in all_results], f, indent=2)
print("\nSaved: embedding_results_summary.csv, embedding_results_summary.json")